In [3]:
import pandas as pd
import requests
import json
from requests.auth import HTTPBasicAuth

# --- CONFIGURATION ---
PLANET_API_KEY = "PLAKa904cfab2d6c4302a360d7bfbc53d6a8"
CSV_FILE = "PLANET_S_36d63cf6273d80f0a332e03615ab13ea[1](in).csv"
ITEM_TYPE = "PSScene"        # 'PSScene' is the standard PlanetScope 3-band / 4-band item type
MAX_CLOUD_COVER = 0.10       # 10% maximum cloud cover
SEARCH_ENDPOINT = "https://api.planet.com/data/v1/quick-search"

# Set up an authenticated session
session = requests.Session()
session.auth = HTTPBasicAuth(PLANET_API_KEY, '')

def search_planet_images(lat, lon, year, max_cloud_cover):
    """
    Builds the API payload and requests scenes matching the criteria for a given year.
    """
    # 1. Geometry Filter (Point of interest)
    geometry_filter = {
        "type": "GeometryFilter",
        "field_name": "geometry",
        "config": {
            "type": "Point",
            "coordinates": [lon, lat] # GeoJSON standard is [Longitude, Latitude]
        }
    }
    
    # 2. Date Range Filter (Entire year)
    date_filter = {
        "type": "DateRangeFilter",
        "field_name": "acquired",
        "config": {
            "gte": f"{year}-01-01T00:00:00.000Z",
            "lte": f"{year}-12-31T23:59:59.000Z"
        }
    }
    
    # 3. Cloud Cover Filter
    cloud_cover_filter = {
        "type": "RangeFilter",
        "field_name": "cloud_cover",
        "config": {
            "lte": max_cloud_cover # 'lte' means Less Than or Equal To
        }
    }
    
    # 4. Combine Filters with an 'AndFilter'
    combined_filter = {
        "type": "AndFilter",
        "config": [geometry_filter, date_filter, cloud_cover_filter]
    }
    
    # 5. Search Request Payload
    search_request = {
        "item_types": [ITEM_TYPE],
        "filter": combined_filter
    }
    
    # 6. Execute POST Request
    response = session.post(SEARCH_ENDPOINT, json=search_request)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

pre_img_ids = []
post_img_ids = []
def main():
    # Load the dataset
    print(f"Loading data from {CSV_FILE}...\n")
    df = pd.read_csv(CSV_FILE)
    
    # Strip any trailing/leading whitespaces from column names (e.g., 'Latitude ' -> 'Latitude')
    df.columns = df.columns.str.strip()
    
    # Iterate through the coordinates
    for index, row in df.iterrows():
        name = row['Data Center']
        lat = row['Latitude']
        lon = row['Longitude']
        pre_year = int(row['Pre-Construction Year'])
        post_year = int(row['Post-Construction Year'])
        
        print(f"--- Processing: {name} ---")
        
        # --- PRE-CONSTRUCTION SEARCH ---
        pre_results = search_planet_images(lat, lon, pre_year, MAX_CLOUD_COVER)
        if pre_results:
            pre_features = pre_results.get('features', [])
            print(f"  [Pre-Construction {pre_year}]: Found {len(pre_features)} low-cloud images.")
            
            # Optional: Print the ID of the clearest image found for the pre-construction year
            if pre_features:
                clearest_pre = min(pre_features, key=lambda x: x['properties']['cloud_cover'])
                print(f"    -> Clearest Image ID: {clearest_pre['id']} (Cloud Cover: {clearest_pre['properties']['cloud_cover']:.2%})")

        # --- POST-CONSTRUCTION SEARCH ---
        post_results = search_planet_images(lat, lon, post_year, MAX_CLOUD_COVER)
        if post_results:
            post_features = post_results.get('features', [])
            print(f"  [Post-Construction {post_year}]: Found {len(post_features)} low-cloud images.")
            
            # Optional: Print the ID of the clearest image found for the post-construction year
            if post_features:
                clearest_post = min(post_features, key=lambda x: x['properties']['cloud_cover'])
                print(f"    -> Clearest Image ID: {clearest_post['id']} (Cloud Cover: {clearest_post['properties']['cloud_cover']:.2%})")
        print("\n")
        # Collect image IDs for further processing
        if pre_features:
            pre_img_ids.extend([feature['id'] for feature in pre_features])
        if post_features:
            post_img_ids.extend([feature['id'] for feature in post_features])

if __name__ == "__main__":
    main()

Loading data from PLANET_S_36d63cf6273d80f0a332e03615ab13ea[1](in).csv...

--- Processing: Apple Mesa Data Center ---
  [Pre-Construction 2014]: Found 1 low-cloud images.
    -> Clearest Image ID: 20141026_181424_0811 (Cloud Cover: 0.00%)
  [Post-Construction 2025]: Found 250 low-cloud images.
    -> Clearest Image ID: 20251212_181357_57_2528 (Cloud Cover: 0.00%)


--- Processing: GDC Phoenix PH1-6 DC ---
  [Pre-Construction 2020]: Found 250 low-cloud images.
    -> Clearest Image ID: 20200802_175952_10_1065 (Cloud Cover: 0.00%)
  [Post-Construction 2025]: Found 250 low-cloud images.
    -> Clearest Image ID: 20251212_181357_57_2528 (Cloud Cover: 0.00%)


--- Processing: Meta Mesa Building 1-5 ---
  [Pre-Construction 2020]: Found 250 low-cloud images.
    -> Clearest Image ID: 20200802_175952_10_1065 (Cloud Cover: 0.00%)
  [Post-Construction 2025]: Found 250 low-cloud images.
    -> Clearest Image ID: 20251212_181357_57_2528 (Cloud Cover: 0.00%)


--- Processing: EdgeCore PH01-05 ---
 

In [4]:
import os
import time
import pandas as pd
import requests
import json
from requests.auth import HTTPBasicAuth

# --- CONFIGURATION ---
PLANET_API_KEY = "PLAKa904cfab2d6c4302a360d7bfbc53d6a8" # Your provided API key
CSV_FILE = "PLANET_S_36d63cf6273d80f0a332e03615ab13ea[1](in).csv"

# Search Config
ITEM_TYPE = "PSScene"        
MAX_CLOUD_COVER = 0.10       
SEARCH_ENDPOINT = "https://api.planet.com/data/v1/quick-search"

# Download Config
PRODUCT_BUNDLE = "visual"    # "visual" for standard viewing, "analytic_sr_udm2" for analysis
DOWNLOAD_DIR = "planet_images"
ORDERS_API_URL = "https://api.planet.com/compute/ops/orders/v2"

# Set up an authenticated session
session = requests.Session()
session.auth = HTTPBasicAuth(PLANET_API_KEY, '')
headers = {'content-type': 'application/json'}

def search_planet_images(lat, lon, year, max_cloud_cover):
    """Builds the API payload and requests scenes matching the criteria for a given year."""
    geometry_filter = {
        "type": "GeometryFilter",
        "field_name": "geometry",
        "config": {
            "type": "Point",
            "coordinates": [lon, lat]
        }
    }
    
    date_filter = {
        "type": "DateRangeFilter",
        "field_name": "acquired",
        "config": {
            "gte": f"{year}-01-01T00:00:00.000Z",
            "lte": f"{year}-12-31T23:59:59.000Z"
        }
    }
    
    cloud_cover_filter = {
        "type": "RangeFilter",
        "field_name": "cloud_cover",
        "config": {
            "lte": max_cloud_cover
        }
    }
    
    combined_filter = {
        "type": "AndFilter",
        "config": [geometry_filter, date_filter, cloud_cover_filter]
    }
    
    search_request = {
        "item_types": [ITEM_TYPE],
        "filter": combined_filter
    }
    
    response = session.post(SEARCH_ENDPOINT, json=search_request)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

def place_order(image_ids):
    """Submits the order to Planet's Orders API."""
    print(f"\nSubmitting order for {len(image_ids)} distinct images...")
    
    order_payload = {
        "name": "Data Center Construction Progression",
        "products": [
            {
                "item_ids": image_ids,
                "item_type": ITEM_TYPE,
                "product_bundle": PRODUCT_BUNDLE
            }
        ]
    }
    
    response = requests.post(
        ORDERS_API_URL, 
        json=order_payload, 
        auth=session.auth, 
        headers=headers
    )
    
    if response.status_code == 202:
        order_info = response.json()
        order_id = order_info['id']
        print(f"Order successfully placed! Order ID: {order_id}")
        return order_info['_links']['_self']
    else:
        print(f"Failed to place order. Status: {response.status_code}")
        print(response.text)
        return None

def download_results(results_links):
    """Downloads the finalized files to the local directory."""
    if not os.path.exists(DOWNLOAD_DIR):
        os.makedirs(DOWNLOAD_DIR)
        
    for result in results_links:
        download_url = result['location']
        filename = result['name']
        filepath = os.path.join(DOWNLOAD_DIR, os.path.basename(filename))
        
        print(f"Downloading {os.path.basename(filename)}...")
        
        with requests.get(download_url, stream=True) as r:
            r.raise_for_status()
            with open(filepath, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
                    
    print(f"\nAll files successfully downloaded to the '{DOWNLOAD_DIR}' folder!")

def poll_and_download(order_url):
    """Polls the API until the order is ready, then triggers the download."""
    print("Polling order status (this may take a few minutes while Planet activates the assets)...")
    
    while True:
        response = requests.get(order_url, auth=session.auth)
        if response.status_code != 200:
            print("Error checking order status.")
            break
            
        order_info = response.json()
        state = order_info['state']
        
        print(f"Current State: {state}")
        
        if state == 'success':
            print("\nOrder is ready! Starting downloads...")
            download_results(order_info['_links']['results'])
            break
        elif state in ['failed', 'partial']:
            print("Order failed or partially failed. Check quota limitations.")
            break
            
        time.sleep(15)

def main():
    print(f"Loading data from {CSV_FILE}...\n")
    try:
        df = pd.read_csv(CSV_FILE)
    except FileNotFoundError:
        print(f"Could not find {CSV_FILE}. Please make sure it's in the same folder.")
        return

    df.columns = df.columns.str.strip()
    
    # We will store the absolute best image IDs here to pass to the Orders API later
    final_image_ids = []
    
    for index, row in df.iterrows():
        name = row['Data Center']
        lat = row['Latitude']
        lon = row['Longitude']
        pre_year = int(row['Pre-Construction Year'])
        post_year = int(row['Post-Construction Year'])
        
        print(f"--- Processing: {name} ---")
        
        # --- PRE-CONSTRUCTION SEARCH ---
        pre_results = search_planet_images(lat, lon, pre_year, MAX_CLOUD_COVER)
        if pre_results:
            pre_features = pre_results.get('features', [])
            print(f"  [Pre-Construction {pre_year}]: Found {len(pre_features)} low-cloud images.")
            
            if pre_features:
                clearest_pre = min(pre_features, key=lambda x: x['properties']['cloud_cover'])
                print(f"    -> Clearest Image ID: {clearest_pre['id']} (Cloud Cover: {clearest_pre['properties']['cloud_cover']:.2%})")
                # Append ONLY the clearest image to our download list
                final_image_ids.append(clearest_pre['id'])

        # --- POST-CONSTRUCTION SEARCH ---
        post_results = search_planet_images(lat, lon, post_year, MAX_CLOUD_COVER)
        if post_results:
            post_features = post_results.get('features', [])
            print(f"  [Post-Construction {post_year}]: Found {len(post_features)} low-cloud images.")
            
            if post_features:
                clearest_post = min(post_features, key=lambda x: x['properties']['cloud_cover'])
                print(f"    -> Clearest Image ID: {clearest_post['id']} (Cloud Cover: {clearest_post['properties']['cloud_cover']:.2%})")
                # Append ONLY the clearest image to our download list
                final_image_ids.append(clearest_post['id'])
        
        print("\n")

    # Remove any duplicate IDs just in case
    final_image_ids = list(set(final_image_ids))

    if not final_image_ids:
        print("No image IDs were found matching your criteria. Aborting download.")
        return

    # --- EXECUTE THE DOWNLOAD ORDER ---
    order_url = place_order(final_image_ids)
    if order_url:
        poll_and_download(order_url)

if __name__ == "__main__":
    main()

Loading data from PLANET_S_36d63cf6273d80f0a332e03615ab13ea[1](in).csv...

--- Processing: Apple Mesa Data Center ---
  [Pre-Construction 2014]: Found 1 low-cloud images.
    -> Clearest Image ID: 20141026_181424_0811 (Cloud Cover: 0.00%)
  [Post-Construction 2025]: Found 250 low-cloud images.
    -> Clearest Image ID: 20251212_181357_57_2528 (Cloud Cover: 0.00%)


--- Processing: GDC Phoenix PH1-6 DC ---
  [Pre-Construction 2020]: Found 250 low-cloud images.
    -> Clearest Image ID: 20200802_175952_10_1065 (Cloud Cover: 0.00%)
  [Post-Construction 2025]: Found 250 low-cloud images.
    -> Clearest Image ID: 20251212_181357_57_2528 (Cloud Cover: 0.00%)


--- Processing: Meta Mesa Building 1-5 ---
  [Pre-Construction 2020]: Found 250 low-cloud images.
    -> Clearest Image ID: 20200802_175952_10_1065 (Cloud Cover: 0.00%)
  [Post-Construction 2025]: Found 250 low-cloud images.
    -> Clearest Image ID: 20251212_181357_57_2528 (Cloud Cover: 0.00%)


--- Processing: EdgeCore PH01-05 ---
 

KeyboardInterrupt: 

In [6]:
import ee
import requests
import pandas as pd
import os

# --- CONFIGURATION ---
CSV_FILE = "/Users/nqj5zk/Library/CloudStorage/OneDrive-UniversityofVirginia/data/AZ/Kathryn_PLANET_img_ids-SDS-LVHQ6LH97F.csv"
DOWNLOAD_DIR = "planet_images" # Saving to your existing folder

# Initialize Earth Engine
# (This will open a browser to authenticate the very first time it is run)
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

def download_landsat_image(name, lat, lon, year):
    print(f"\nSearching Landsat 5 for {name} ({year})...")
    point = ee.Geometry.Point([lon, lat])
    
    # Landsat 5 TM Collection (Top of Atmosphere)
    L5_COLLECTION = "LANDSAT/LT05/C02/T1_TOA"
    
    # Filter for the exact year, coordinate, and lowest cloud cover
    dataset = (ee.ImageCollection(L5_COLLECTION)
               .filterBounds(point)
               .filterDate(f"{year}-01-01", f"{year}-12-31")
               .filter(ee.Filter.lt('CLOUD_COVER', 10))
               .sort('CLOUD_COVER'))
    
    count = dataset.size().getInfo()
    
    # Fallback: If 10% cloud cover is too strict, expand to 30%
    if count == 0:
        print("  -> No images under 10% cloud cover. Expanding search to 30%...")
        dataset = (ee.ImageCollection(L5_COLLECTION)
                   .filterBounds(point)
                   .filterDate(f"{year}-01-01", f"{year}-12-31")
                   .filter(ee.Filter.lt('CLOUD_COVER', 30))
                   .sort('CLOUD_COVER'))
        count = dataset.size().getInfo()
        if count == 0:
             print("  -> Still no low-cloud images found for this year. Skipping.")
             return

    # Select the absolute clearest image from that dataset
    image = ee.Image(dataset.first())
    cloud_cover = image.get('CLOUD_COVER').getInfo()
    date = image.get('DATE_ACQUIRED').getInfo()
    print(f"  -> Found clearest image from {date} (Cloud Cover: {cloud_cover:.2f}%)")

    # Apply Landsat 5 visual parameters (Band 3=Red, Band 2=Green, Band 1=Blue)
    rgb_image = image.visualize(bands=['B3', 'B2', 'B1'], min=0.0, max=0.4)

    # Create a 5km buffer around the coordinate to generate a 10km x 10km viewing window
    buffer = point.buffer(5000).bounds()

    # Generate the download URL and save the TIF
    try:
        safe_name = name.replace(' ', '_').replace('/', '-')
        url = rgb_image.getDownloadURL({
            'name': f"{safe_name}_{year}",
            'scale': 30, # 30 meters/pixel (Landsat native)
            'region': buffer,
            'format': 'GEO_TIFF'
        })
        
        print("  -> Downloading TIF file...")
        response = requests.get(url)
        filepath = os.path.join(DOWNLOAD_DIR, f"Landsat_{safe_name}_{year}_Visual.tif")
        
        with open(filepath, 'wb') as f:
            f.write(response.content)
        print(f"  -> Successfully saved to {filepath}")
        
    except Exception as e:
        print(f"  -> Error downloading {name}: {e}")

def main():
    print(f"Loading data from {CSV_FILE}...")
    try:
        df = pd.read_csv(CSV_FILE)
    except FileNotFoundError:
        print(f"Could not find {CSV_FILE}. Make sure it is in the same directory.")
        return
        
    df.columns = df.columns.str.strip()
    
    # Filter for locations where Pre-Construction Year is 2010 or older
    missing_df = df[df['Pre-Construction Year'] <= 2010]
    
    if not os.path.exists(DOWNLOAD_DIR):
        os.makedirs(DOWNLOAD_DIR)
        
    print(f"Found {len(missing_df)} locations needing historical Landsat imagery.\n")
    
    for index, row in missing_df.iterrows():
        name = row['Data Center']
        lat = row['Latitude']
        lon = row['Longitude']
        pre_year = int(row['Pre-Construction Year'])
        
        download_landsat_image(name, lat, lon, pre_year)
        
    print("\nAll historical GEE downloads complete!")

if __name__ == "__main__":
    main()


Successfully saved authorization token.
Loading data from /Users/nqj5zk/Library/CloudStorage/OneDrive-UniversityofVirginia/data/AZ/Kathryn_PLANET_img_ids-SDS-LVHQ6LH97F.csv...
Found 5 locations needing historical Landsat imagery.


Searching Landsat 5 for Cyprus One PHX1-8 (2010)...
  -> Found clearest image from 2010-03-15 (Cloud Cover: 0.00%)
  -> Downloading TIF file...
  -> Successfully saved to planet_images/Landsat_Cyprus_One_PHX1-8_2010_Visual.tif

Searching Landsat 5 for H5 Data Centers Phoenix (2010)...
  -> Found clearest image from 2010-03-15 (Cloud Cover: 0.00%)
  -> Downloading TIF file...
  -> Successfully saved to planet_images/Landsat_H5_Data_Centers_Phoenix_2010_Visual.tif

Searching Landsat 5 for Switch Las Vegas 7-12, 15 (2004)...
  -> Found clearest image from 2004-06-07 (Cloud Cover: 0.00%)
  -> Downloading TIF file...
  -> Successfully saved to planet_images/Landsat_Switch_Las_Vegas_7-12,_15_2004_Visual.tif

Searching Landsat 5 for Switch Las Vegas 2-6 (2000)...
